# Reference

1. Life visualization, from [here](https://towardsdatascience.com/how-to-visualize-the-rest-of-your-life-28f943b1f70b)  (inspired by Kurzgesagt).

2. Web app [here](https://your-ilfe-in-weeks.vercel.app/) (in comments from point 1)

3. Interactive [here](https://labs.coruscantconsulting.co.uk/life/weeks/) (I found this after writing this code)

# Code

## Python/Jupyter Notebook (+ Altair)

This is the original version, which allows for interactivity but not for web deployment.

Import libraries

In [4]:
import pandas as pd
import math
from datetime import date, datetime, timedelta
import altair as alt

# import widgets for interactivity
import ipywidgets as widgets

# create useful constants
c_label_lived = "lived weeks"
c_label_remaining = "remaining weeks"
c_label_current = "this week"

c_width = 1080 / 2.2
c_height = 1500 / 2.2

Personal stats and life expectancy

(Disabled since this widget doesn't work in VSCode)

birthday = input(widgets.DatePicker(
    description='Birthday:', 
    disabled=True,
    ))

In [5]:
birthday = input("What's your birthday? Enter in YYYY-MM-DD format")

Import life expectancy data:

Filtered data from the WHO [here](https://apps.who.int/gho/data/node.main.688), using 2019 life expectancy data.

_This isn't 100% accurate because those born before 2020 will have a different life expectancy (usually lower, but not always)._

_Downloaded as a .json because I want to learn how to work with .json files_

Method below from [here](https://towardsdatascience.com/how-to-convert-json-into-a-pandas-dataframe-100b2ae1e0d8), point #4

In [6]:
import json
# load data using Python JSON module
with open('data.json','r') as f:
    data = json.loads(f.read())
    
# Normalizing data
df = pd.json_normalize(data, record_path =['fact'])

In [7]:
# create a list from the `dims.COUNTRY` column, to make a drop-down list
countries_full = df["dims.COUNTRY"].tolist()
# remove duplicates, while keeping the original order
countries = []
for i in countries_full:
    if i not in countries:
        countries.append(i)

In [8]:
# Allow selection of country of birth
country_selector = widgets.Dropdown(
    placeholder='Your country of birth',
    options=countries,
    value = "United States of America",
    description='Country:',
    ensure_option=True,
)
country_selector

Dropdown(description='Country:', index=174, options=('Afghanistan', 'Albania', 'Algeria', 'Angola', 'Antigua a…

In [9]:
# Get value from selection from `country selector` above, to use to search for life expectancy by selected country
selected_country = country_selector.value
selected_country

'United States of America'

In [10]:
# Allow selection of gender
gender_selector = widgets.Dropdown(
    placeholder='Your gender',
    options=['Male','Female','Both sexes'],
    value = "Male",
    description='Gender'
)
gender_selector

Dropdown(description='Gender', options=('Male', 'Female', 'Both sexes'), value='Male')

In [11]:
selected_gender = gender_selector.value

In [12]:
life_exp = float(df.loc[(df['dims.COUNTRY']==selected_country) & (df['dims.SEX']==selected_gender),'Value'].values[0])

In [13]:
life_exp

76.3

In [14]:
birthday_val = datetime.strptime(birthday, '%Y-%m-%d')

In [15]:
life = timedelta(life_exp * 365.25)

In [16]:
current_date = datetime.today()
day_of_birth = birthday_val
day_of_death = birthday_val + life
day_of_death

datetime.datetime(2069, 10, 27, 13, 48)

Calculate time from birth until now, time from now until death.

In [17]:
lived_life = current_date - day_of_birth
rest_of_life = day_of_death - current_date

lived_life_years = (lived_life.days / 365.25)
lived_life_years_floor = math.floor(lived_life_years) # .floor = Round numbers down to the nearest integer

lived_life_weeks = (lived_life_years - lived_life_years_floor) * 365.25 / 7
lived_life_weeks_floor = math.floor(lived_life_weeks)

rest_of_life_years = (rest_of_life.days / 365.25)
rest_of_life_years_floor = math.floor(rest_of_life_years)

Create three data frames, to hold information about each week, whether it has already passed, whether it is the current week, or a future week.

In [18]:
df_ll_weeks = pd.DataFrame(columns = ['week', 'year', 'label'])

for week in range (lived_life_weeks_floor):
    df_ll_weeks = df_ll_weeks.append(pd.DataFrame({
        'week':[week],
        'year':[lived_life_years_floor],
        'label':[c_label_lived]
    }))

df_ll_weeks = df_ll_weeks.append(pd.DataFrame({
    'week':[lived_life_weeks_floor],
    'year':[lived_life_years_floor],
    'label':[c_label_current]
}))

for week in range (lived_life_weeks_floor + 1, 52):
    df_ll_weeks = df_ll_weeks.append(pd.DataFrame({
        'week':[week],
        'year':[lived_life_years_floor],
        'label':[c_label_remaining]
    }))

df_ll = pd.DataFrame(columns = ['week', 'year', 'label'])

for year in range(0, lived_life_years_floor + 0):
    for week in range(52):
        df_ll = df_ll.append(pd.DataFrame({
            'week':[week],
            'year':[year],
            'label':[c_label_lived]
        }))

df_rl = pd.DataFrame(columns = ['week', 'year', 'label'])

for year in range(lived_life_years_floor + 1, lived_life_years_floor + rest_of_life_years_floor + 1):
    for week in range(52):
        df_rl = df_rl.append(pd.DataFrame({
            'week':[week],
            'year':[year],
            'label':[c_label_remaining]
        }))

Create the chart

In [19]:
chart = (
    alt.Chart(pd.concat([df_ll, df_rl, df_ll_weeks]))
    .mark_square(
        filled = True,
        opacity = 1,
        # color = "black",
        size = 8
    ).encode(
        x = alt.X("week", axis = None),
        y = alt.Y("year", axis = None),
        color = alt.Color(
            "label", scale = alt.Scale(range = ["black", "lightgrey", "red"]),
            legend = alt.Legend(orient = "bottom"), title = ""
        ),
        tooltip=["year"]
    ).properties(
        width = c_width,
        height = c_height
    ).properties(
        title = "Your Life in Weeks"
        
    )
)

Configure the chart

In [20]:
chart_config = (
    chart
    .configure_title(
        fontSize = 40,
        font = "Arial",
        align = "center",
        color = "black",
        baseline = "bottom",
        dy = 36
    ).configure_view(
        strokeWidth = 0
    )
)
display(chart_config)
chart_config.save("rest_of_life.html")

alt.Chart(...)

---

## Bokeh

Trying for deployment on GitHub page. Bokeh and Panel seem to be able to do this

In [40]:
import bokeh

In [41]:
import panel as pn